In [1]:
import numpy as np
import sys
import types
import os
# =========================================================
# 0. HARD DISABLE MPI (ROBUST VERSION)
# =========================================================
import os
import sys
import types

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["DESI_DISABLE_MPI"] = "1"

# -------------------------
# Fake MPI implementation
# -------------------------
class FakeComm:
    def Get_rank(self): return 0
    def Get_size(self): return 1
    def Barrier(self): pass

class FakeMPI:
    COMM_WORLD = FakeComm()
    COMM_SELF = FakeComm()
    ANY_SOURCE = -1

# -------------------------
# Fake mpi4py module tree
# -------------------------
mpi4py = types.ModuleType("mpi4py")
mpi4py.MPI = FakeMPI

sys.modules["mpi4py"] = mpi4py
sys.modules["mpi4py.MPI"] = FakeMPI

# IMPORTANT: also fake direct import style
sys.modules["MPI"] = FakeMPI



# ---------------------------
# NOW safe to import desilike
# ---------------------------
from desilike.theories.galaxy_clustering import BAOTheoryCorrelationFunction
from desilike.observables import Observable
from desilike.likelihoods import ObservablesGaussianLikelihood
from desilike.profilers import MinuitProfiler


# ---------------------------
# Load data
# ---------------------------
filename = "../data/monopoles/box/mag=-21.2_sep=1.0-150.0_binsep=2.0_full.npz"

data = np.load(filename)
print("keys:", data.files)

s = data["s"]
xi = data["xi"]


# ---------------------------
# Fake covariance (temporary)
# ---------------------------
sigma_xi = 0.05 * np.abs(xi)
cov = np.diag(sigma_xi**2)


# ---------------------------
# Theory
# ---------------------------
theory = BAOTheoryCorrelationFunction()
theory.init.update(mode='monopole', s=s)

params = theory.params
params['alpha'].update(value=1.0, prior={'limits': [0.8, 1.2]})
params['Sigma_nl'].update(value=5.0, prior={'limits': [0., 20.]})

# broadband
for name in params.select(basename='a*'):
    params[name].update(fixed=False)


# ---------------------------
# Likelihood
# ---------------------------
observable = Observable(data=xi, covariance=cov, theory=theory)
likelihood = ObservablesGaussianLikelihood(observables=[observable])


# ---------------------------
# Fit
# ---------------------------
profiler = MinuitProfiler(likelihood)
bestfit = profiler.maximize()


print("\nalpha =", bestfit.params['alpha'])
print("Sigma_nl =", bestfit.params['Sigma_nl'])
print("chi2 =", bestfit.chi2)

AttributeError: type object 'FakeMPI' has no attribute 'ANY_TAG'